In [ ]:
import os
import random
from datetime import datetime, timedelta

from faker import Faker
from great_expectations.exceptions import DataContextError
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, when, col
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, StringType

import great_expectations as gx

In [ ]:
schema = StructType([
    StructField("id", LongType(), False),
    StructField("published", TimestampType(), True),
    StructField("subject", StringType(), True),
    StructField("keyword", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("description", StringType(), True),
    StructField("original_link", StringType(), True),
    StructField("link", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
])

In [ ]:
def random_dt(within_days=30):
    return datetime.now() - timedelta(
        days=random.randint(0, within_days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )


def generate_news_data(fake: Faker, n: int, start_id: int = 1):
    return [generate_news_row(fake, start_id + i) for i in range(n)]


def generate_news_row(fake: Faker, id_: int):
    published = random_dt(14)
    return {
        "id": id_,
        "published": published,
        "subject": random.choice(["경제", "사회", "정치", "IT", "국제", "문화"]),
        "keyword": ", ".join(fake.words(nb=random.randint(2, 6)))[:200],
        "title": fake.sentence(nb_words=12)[:1000],
        "summary": fake.text(max_nb_chars=600)[:4000],
        "description": fake.text(max_nb_chars=2500),
        "original_link": fake.url()[:500],
        "link": fake.url()[:500],
        "created_at": published,
        "updated_at": published,
    }

In [ ]:
spark = SparkSession.builder.appName("news-dummy").master("spark://spark-master.mmix.io:7077").config("spark.sql.shuffle.partitions", "1").getOrCreate()
news = spark.createDataFrame(generate_news_data(Faker("ko_KR"), 10), schema=schema)
news_bad = news.withColumn("title", when(col("id") % 200 == 0, lit(None)).otherwise(col("title"))).withColumn("link", when(col("id") % 333 == 0, lit("not-a-url")).otherwise(col("link")))

In [ ]:
#context = gx.get_context(mode="file", context_root_dir="/Users/genius/Workspace/enjoy-workreduce/notebooks/spark/great_expectations")
context = gx.get_context()

In [ ]:
DATASOURCE_NAME = "spark_ds"
ASSET_NAME = "runtime_df"
BATCH_DEF_NAME = "whole_df"

if DATASOURCE_NAME in context.data_sources.all():
    spark_ds = context.data_sources.get(DATASOURCE_NAME)
else:
    spark_ds = context.data_sources.add_spark(name=DATASOURCE_NAME)

if ASSET_NAME in spark_ds.get_asset_names():
    df_asset = spark_ds.get_asset(ASSET_NAME)
else:
    df_asset = spark_ds.add_dataframe_asset(name=ASSET_NAME)

try:
    batch_def = df_asset.get_batch_definition(BATCH_DEF_NAME)
except Exception:
    batch_def = df_asset.add_batch_definition_whole_dataframe(BATCH_DEF_NAME)

batch_parameters = {"dataframe": news_bad}

In [ ]:
SUITE_NAME = "spark_df_suite"
try:
    suite = context.suites.get(SUITE_NAME)
except Exception:
    suite = gx.ExpectationSuite(name=SUITE_NAME)
    suite = context.suites.add(suite)

suite.add_expectation(gx.expectations.ExpectColumnToExist(column="user_id"))
suite.save()

In [ ]:
VD_NAME = "spark_df_validation_definition"
try:
    validation_definition = context.validation_definitions.get(VD_NAME)
except DataContextError:
    validation_definition = gx.ValidationDefinition(name=VD_NAME, data=batch_def, suite=suite)
    validation_definition = context.validation_definitions.add(validation_definition)

In [ ]:
validation_results = validation_definition.run(batch_parameters=batch_parameters, result_format={"result_format": "SUMMARY"})

In [ ]:
from great_expectations.checkpoint.actions import _VALIDATION_ACTION_REGISTRY

print(sorted(_VALIDATION_ACTION_REGISTRY._registered_actions.keys()))

In [ ]:
from great_expectations.checkpoint import Checkpoint
from great_expectations.exceptions import DataContextError

CHECKPOINT_NAME = "spark_runtime_checkpoint"
VD_NAME = "spark_df_validation_definition"

try:
    checkpoint = context.checkpoints.get(CHECKPOINT_NAME)
except DataContextError:
    checkpoint = Checkpoint(
        name=CHECKPOINT_NAME,
        validation_definitions=[{"name": VD_NAME}],
        actions=[
            {
                "name": "update_data_docs",
                "type": "update_data_docs",
                "site_names": ["minio_site"],
            }
        ],
    )
    checkpoint = context.checkpoints.add(checkpoint)

result = checkpoint.run(batch_parameters=batch_parameters)
print(result.describe())


In [ ]:
print("validation_results_store_name:", context.validation_results_store_name)
print("backend:", context.stores[context.validation_results_store_name].store_backend)